# LeetCode 1248 — Count Number of Nice Subarrays bằng Python

Link đề: https://leetcode.com/problems/count-number-of-nice-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Sliding Window, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def numberOfSubarrays(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: "subarray" là **mảng con liên tiếp**; đếm số mảng con chứa **đúng** `k` số lẻ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biến đổi bài toán: thay mỗi phần tử bằng `nums[i] % 2` (1 nếu lẻ, 0 nếu chẵn) → bài trở thành "đếm mảng con có **tổng đúng bằng `k`**" trên mảng 0/1.
2. Dùng **tổng tiền tố** (prefix sum) + **bảng băm đếm** (hash map) để đếm trong `O(n)` — kỹ thuật của bài 560 Subarray Sum Equals K.
3. Dùng **cửa sổ trượt** (sliding window) với mẹo `exactly(k) = atMost(k) − atMost(k − 1)`.
4. Đếm bằng **toán tổ hợp** trên vị trí các số lẻ: số cách mở rộng sang trái × số cách mở rộng sang phải.
5. Tránh các lỗi: quên `count[0] = 1`, cộng giá trị thay vì tính chẵn lẻ, cửa sổ "đúng k" bỏ sót số chẵn ở đầu.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Một mảng con liên tiếp được gọi là **nice** (đẹp) nếu nó chứa **đúng `k` số lẻ**. Trả về số mảng con đẹp.

**Ví dụ 1:** `nums = [1,1,2,1,1], k = 3` → `2` (chỉ có `[1,1,2,1]` và `[1,2,1,1]`).

**Ví dụ 2:** `nums = [2,4,6], k = 1` → `0` (không có số lẻ nào).

**Ví dụ 3:** `nums = [2,2,2,1,2,2,1,2,2,2], k = 2` → `16`.

**Ràng buộc:** `1 <= nums.length <= 50000`; `1 <= nums[i] <= 10⁵`; `1 <= k <= nums.length`.

In [1]:
import bisect
import random
import time
from collections import Counter, defaultdict
from itertools import accumulate
from typing import List

examples = [
    ([1, 1, 2, 1, 1], 3, 2),
    ([2, 4, 6], 1, 0),
    ([2, 2, 2, 1, 2, 2, 1, 2, 2, 2], 2, 16),
]
for nums, k, want in examples:
    print(nums, "k =", k, "->", want)

[1, 1, 2, 1, 1] k = 3 -> 2
[2, 4, 6] k = 1 -> 0
[2, 2, 2, 1, 2, 2, 1, 2, 2, 2] k = 2 -> 16


## 3. Kiến thức nền

### 3.1 Kiểm tra chẵn lẻ: `x % 2` và `x & 1`

`x % 2 == 1` (hoặc `x & 1 == 1`) ⇔ `x` lẻ. Biến mảng thành mảng 0/1 giúp "đếm số lẻ" trở thành "tính tổng".

In [2]:
nums = [2, 2, 2, 1, 2, 2, 1, 2, 2, 2]
bits = [x % 2 for x in nums]
print(bits, "| số lẻ trong cả mảng =", sum(bits))
print([x & 1 for x in [7, 8, 100001]])

[0, 0, 0, 1, 0, 0, 1, 0, 0, 0] | số lẻ trong cả mảng = 2
[1, 0, 1]


### 3.2 Mảng con liên tiếp và số lượng của chúng

Mảng `n` phần tử có `n(n+1)/2` mảng con liên tiếp (chọn đầu `i ≤` cuối `j`). Với `n = 5·10⁴` là ~`1.25·10⁹` — không thể duyệt hết.

In [3]:
n = 4
subs = [(i, j) for i in range(n) for j in range(i, n)]
print(len(subs), n * (n + 1) // 2, subs[:6])

10 10 [(0, 0), (0, 1), (0, 2), (0, 3), (1, 1), (1, 2)]


### 3.3 Tổng tiền tố (prefix sum)

`pre[0] = 0`, `pre[i+1] = pre[i] + a[i]`. Tổng đoạn `a[i..j]` bằng `pre[j+1] − pre[i]`. `itertools.accumulate` tính nhanh dãy tổng tích luỹ.

In [4]:
a = [1, 1, 0, 1, 1]
pre = [0] + list(accumulate(a))
print("pre =", pre)
i, j = 1, 3
print(f"tổng a[{i}..{j}] =", sum(a[i:j + 1]), "=", pre[j + 1] - pre[i])

pre = [0, 1, 2, 2, 3, 4]
tổng a[1..3] = 2 = 2


### 3.4 Đếm tần suất với `Counter` / `defaultdict(int)`

Khi duyệt prefix từ trái sang phải, ta cần hỏi "đã có bao nhiêu prefix trước đó bằng giá trị `v`?" → bảng đếm `cnt[v]`.

In [5]:
cnt = Counter()
for v in [0, 1, 2, 2, 3, 4]:
    print(f"gặp {v}: trước đó đã có {cnt[v]} lần")
    cnt[v] += 1

gặp 0: trước đó đã có 0 lần
gặp 1: trước đó đã có 0 lần
gặp 2: trước đó đã có 0 lần
gặp 2: trước đó đã có 1 lần
gặp 3: trước đó đã có 0 lần
gặp 4: trước đó đã có 0 lần


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `nums = [1,1,2,1,1]`, `k = 3`. Mảng 0/1: `[1,1,0,1,1]`. Prefix: `pre = [0,1,2,2,3,4]`.

Đoạn `nums[i..j]` có đúng 3 số lẻ ⇔ `pre[j+1] − pre[i] = 3`. Liệt kê các cặp `(i, j+1)` với `i < j+1` và hiệu bằng 3:

- `pre[4] = 3`, `pre[0] = 0` → đoạn `[0..3]` = `[1,1,2,1]` ✓
- `pre[5] = 4`, `pre[1] = 1` → đoạn `[1..4]` = `[1,2,1,1]` ✓

Không còn cặp nào khác → đáp án `2`. Như vậy: **với mỗi vị trí kết thúc, đếm số prefix trước đó bằng `pre hiện tại − k`**.

In [6]:
nums, k = [1, 1, 2, 1, 1], 3
pre = [0] + list(accumulate(x % 2 for x in nums))
pairs = [(i, e) for e in range(len(pre)) for i in range(e) if pre[e] - pre[i] == k]
print("pre =", pre)
print("cặp (i, j+1):", pairs, "->", [nums[i:e] for i, e in pairs])

pre = [0, 1, 2, 2, 3, 4]
cặp (i, j+1): [(0, 4), (1, 5)] -> [[1, 1, 2, 1], [1, 2, 1, 1]]


## 5. Cách 1 — Brute Force / baseline

Với mỗi điểm đầu `i`, mở rộng điểm cuối `j` và cộng dồn số lẻ; tăng đáp án khi số lẻ bằng `k`, dừng sớm khi vượt `k` (số lẻ chỉ tăng khi `j` tăng).

In [7]:
class SolutionBrute:
    def numberOfSubarrays(self, nums, k):
        n = len(nums)
        ans = 0
        for i in range(n):
            odd = 0
            for j in range(i, n):
                odd += nums[j] & 1
                if odd == k:
                    ans += 1
                elif odd > k:
                    break
        return ans


for nums, k, want in examples:
    print(SolutionBrute().numberOfSubarrays(nums, k), want)

2 2
0 0
16 16


**Độ phức tạp:** `O(n²)` worst (ví dụ mảng toàn số chẵn: không bao giờ `break`) → `2.5·10⁹` với `n = 5·10⁴`: quá chậm. Bộ nhớ `O(1)`.

**Nút thắt:** với mỗi `i`, ta đếm lại từ đầu. Câu hỏi "có bao nhiêu `i` hợp với `j` này" có thể trả lời bằng tra bảng.

## 6. Bước đột phá

**Insight 1 (prefix + hash map):** đặt `p_j` = số lẻ trong `nums[0..j-1]`. Đoạn `nums[i..j-1]` đẹp ⇔ `p_j − p_i = k` ⇔ `p_i = p_j − k`. Duyệt `j` từ trái sang phải, giữ `cnt[v]` = số chỉ số `i < j` có `p_i = v`; cộng `cnt[p_j − k]` vào đáp án. Khởi tạo `cnt[0] = 1` (prefix rỗng).

**Insight 2 (atMost):** "số mảng con có **nhiều nhất** `k` số lẻ" đếm dễ bằng cửa sổ trượt (cửa sổ hợp lệ co giãn đơn điệu). Khi đó `đúng k = atMost(k) − atMost(k − 1)`.

**Insight 3 (tổ hợp vị trí):** gọi `pos` là vị trí các số lẻ. Một mảng con đẹp chứa **đúng** các số lẻ `pos[t], …, pos[t+k−1]`. Điểm đầu nằm trong `(pos[t−1], pos[t]]` (có `pos[t] − pos[t−1]` lựa chọn), điểm cuối trong `[pos[t+k−1], pos[t+k])` (có `pos[t+k] − pos[t+k−1]` lựa chọn), với lính canh `pos[−1] = −1`, `pos[m] = n`. Đáp án = tổng các tích.

**Invariant cho Insight 1:** trước khi xử lý `j`, `cnt` chứa đúng tần suất của `p_0, …, p_{j−1}`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix sum + hash map

In [8]:
class SolutionPrefixHash:
    def numberOfSubarrays(self, nums, k):
        cnt = Counter({0: 1})
        odd = ans = 0
        for x in nums:
            odd += x & 1
            ans += cnt[odd - k]
            cnt[odd] += 1
        return ans


for nums, k, want in examples:
    print(SolutionPrefixHash().numberOfSubarrays(nums, k), want)

2 2
0 0
16 16


Vì prefix số lẻ chỉ nằm trong `[0, n]`, ta có thể dùng **mảng** `cnt = [0] * (n + 1)` thay cho `Counter` — nhanh hơn một chút (đây là lời giải phổ biến):

In [9]:
class SolutionPrefixArray:
    def numberOfSubarrays(self, nums, k):
        cnt = [0] * (len(nums) + 1)
        cnt[0] = 1
        odd = ans = 0
        for x in nums:
            odd += x & 1
            if odd >= k:
                ans += cnt[odd - k]
            cnt[odd] += 1
        return ans


for nums, k, want in examples:
    print(SolutionPrefixArray().numberOfSubarrays(nums, k), want)

2 2
0 0
16 16


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.2 Cách 3 — Sliding window: `atMost(k) − atMost(k − 1)`

`atMost(K)`: với mỗi điểm cuối `r`, co `l` cho tới khi cửa sổ có ≤ `K` số lẻ; mọi điểm đầu trong `[l, r]` đều hợp lệ → cộng `r − l + 1`.

In [10]:
class SolutionAtMost:
    def numberOfSubarrays(self, nums, k):
        def at_most(K):
            if K < 0:
                return 0
            l = odd = res = 0
            for r, x in enumerate(nums):
                odd += x & 1
                while odd > K:
                    odd -= nums[l] & 1
                    l += 1
                res += r - l + 1
            return res

        return at_most(k) - at_most(k - 1)


for nums, k, want in examples:
    print(SolutionAtMost().numberOfSubarrays(nums, k), want)

2 2
0 0
16 16


**Độ phức tạp:** mỗi lần `at_most` có `l`, `r` chỉ tiến → `O(n)`; hai lần → `O(n)`. Bộ nhớ `O(1)`.

### 7.3 Cách 4 — Tổ hợp trên vị trí số lẻ

In [11]:
class SolutionPositions:
    def numberOfSubarrays(self, nums, k):
        pos = [-1] + [i for i, x in enumerate(nums) if x & 1] + [len(nums)]
        ans = 0
        # các số lẻ thật nằm ở pos[1..m]; nhóm k số lẻ liên tiếp bắt đầu tại pos[t]
        for t in range(1, len(pos) - k):
            left = pos[t] - pos[t - 1]
            right = pos[t + k] - pos[t + k - 1]
            ans += left * right
        return ans


for nums, k, want in examples:
    print(SolutionPositions().numberOfSubarrays(nums, k), want)

2 2
0 0
16 16


Kiểm tra ví dụ 3: `pos = [-1, 3, 6, 10]`, `k = 2`, chỉ có `t = 1`: `left = 3 − (−1) = 4`, `right = 10 − 6 = 4` → `16` ✓.

**Độ phức tạp:** `O(n)` thời gian, `O(m)` bộ nhớ (`m` = số số lẻ).

### 7.4 Cách 5 — Sliding window một lượt, đếm "số chẵn dẫn đầu"

Duy trì cửa sổ có **đúng** `k` số lẻ. Khi cửa sổ vừa đủ `k` số lẻ (do `r` mới), đẩy `l` qua các số chẵn đầu cửa sổ và đếm `prefix_evens + 1` = số điểm đầu hợp lệ; với các `r` tiếp theo là số chẵn, số điểm đầu hợp lệ **không đổi**, nên cộng lại cùng giá trị.

In [12]:
class SolutionOnePass:
    def numberOfSubarrays(self, nums, k):
        l = odd = ans = 0
        ways = 0                         # số điểm đầu hợp lệ cho điểm cuối hiện tại
        for x in nums:
            if x & 1:
                odd += 1
                ways = 0
            if odd > k:                  # bỏ số lẻ đầu tiên của cửa sổ
                while nums[l] % 2 == 0:
                    l += 1
                l += 1
                odd -= 1
            if odd == k and ways == 0:   # vừa đủ k số lẻ (hoặc vừa thay đổi): đếm lại
                while nums[l] % 2 == 0:
                    ways += 1
                    l += 1
                ways += 1                # điểm đầu ngay tại số lẻ đầu tiên
            if odd == k:
                ans += ways
        return ans


for nums, k, want in examples:
    print(SolutionOnePass().numberOfSubarrays(nums, k), want)

2 2
0 0
16 16


Lưu ý: sau khi đếm `ways`, `l` đang đứng **tại** số lẻ đầu tiên của cửa sổ; lần "bỏ số lẻ đầu" tiếp theo sẽ bước qua nó (vòng `while` không chạy vì `nums[l]` lẻ), rồi `l += 1`. Cách này `O(n)`, `O(1)` bộ nhớ, nhưng dễ sai biên — cần test kỹ (mục 12).

## 8. TRACE từng bước

Trace Cách 2 (prefix + hash map) trên ví dụ 3, `k = 2`:

| j | x | odd | cần `odd − k` | `cnt[odd−k]` | ans | cnt sau |
|---|---|---|---|---|---|---|
| 0 | 2 | 0 | −2 | 0 | 0 | {0:2} |
| 1 | 2 | 0 | −2 | 0 | 0 | {0:3} |
| 2 | 2 | 0 | −2 | 0 | 0 | {0:4} |
| 3 | 1 | 1 | −1 | 0 | 0 | {0:4,1:1} |
| 4 | 2 | 1 | −1 | 0 | 0 | {0:4,1:2} |
| 5 | 2 | 1 | −1 | 0 | 0 | {0:4,1:3} |
| 6 | 1 | 2 | 0 | 4 | 4 | {…,2:1} |
| 7 | 2 | 2 | 0 | 4 | 8 | {…,2:2} |
| 8 | 2 | 2 | 0 | 4 | 12 | {…,2:3} |
| 9 | 2 | 2 | 0 | 4 | 16 | {…,2:4} |

In [13]:
def trace_prefix(nums, k):
    cnt = Counter({0: 1})
    odd = ans = 0
    for j, x in enumerate(nums):
        odd += x & 1
        add = cnt[odd - k]
        ans += add
        cnt[odd] += 1
        print(f"j={j}: x={x} odd={odd} cần {odd - k:2} -> +{add} | ans={ans:2} | cnt={dict(cnt)}")
    return ans


trace_prefix([2, 2, 2, 1, 2, 2, 1, 2, 2, 2], 2)

j=0: x=2 odd=0 cần -2 -> +0 | ans= 0 | cnt={0: 2}
j=1: x=2 odd=0 cần -2 -> +0 | ans= 0 | cnt={0: 3}
j=2: x=2 odd=0 cần -2 -> +0 | ans= 0 | cnt={0: 4}
j=3: x=1 odd=1 cần -1 -> +0 | ans= 0 | cnt={0: 4, 1: 1}
j=4: x=2 odd=1 cần -1 -> +0 | ans= 0 | cnt={0: 4, 1: 2}
j=5: x=2 odd=1 cần -1 -> +0 | ans= 0 | cnt={0: 4, 1: 3}
j=6: x=1 odd=2 cần  0 -> +4 | ans= 4 | cnt={0: 4, 1: 3, 2: 1}
j=7: x=2 odd=2 cần  0 -> +4 | ans= 8 | cnt={0: 4, 1: 3, 2: 2}
j=8: x=2 odd=2 cần  0 -> +4 | ans=12 | cnt={0: 4, 1: 3, 2: 3}
j=9: x=2 odd=2 cần  0 -> +4 | ans=16 | cnt={0: 4, 1: 3, 2: 4}


16

Trace Cách 3 (`atMost`) trên ví dụ 1 để thấy cửa sổ co giãn:

In [14]:
def trace_at_most(nums, K):
    l = odd = res = 0
    for r, x in enumerate(nums):
        odd += x & 1
        while odd > K:
            odd -= nums[l] & 1
            l += 1
        res += r - l + 1
        print(f"  K={K} r={r}: cửa sổ [{l},{r}] = {nums[l:r + 1]}, odd={odd}, +{r - l + 1} -> {res}")
    return res


a3 = trace_at_most([1, 1, 2, 1, 1], 3)
a2 = trace_at_most([1, 1, 2, 1, 1], 2)
print("atMost(3) - atMost(2) =", a3, "-", a2, "=", a3 - a2)

  K=3 r=0: cửa sổ [0,0] = [1], odd=1, +1 -> 1
  K=3 r=1: cửa sổ [0,1] = [1, 1], odd=2, +2 -> 3
  K=3 r=2: cửa sổ [0,2] = [1, 1, 2], odd=2, +3 -> 6
  K=3 r=3: cửa sổ [0,3] = [1, 1, 2, 1], odd=3, +4 -> 10
  K=3 r=4: cửa sổ [1,4] = [1, 2, 1, 1], odd=3, +4 -> 14
  K=2 r=0: cửa sổ [0,0] = [1], odd=1, +1 -> 1
  K=2 r=1: cửa sổ [0,1] = [1, 1], odd=2, +2 -> 3
  K=2 r=2: cửa sổ [0,2] = [1, 1, 2], odd=2, +3 -> 6
  K=2 r=3: cửa sổ [1,3] = [1, 2, 1], odd=2, +3 -> 9
  K=2 r=4: cửa sổ [2,4] = [2, 1, 1], odd=2, +3 -> 12
atMost(3) - atMost(2) = 14 - 12 = 2


## 9. Code nộp LeetCode

In [15]:
from typing import List


class Solution:
    def numberOfSubarrays(self, nums: List[int], k: int) -> int:
        cnt = [0] * (len(nums) + 1)
        cnt[0] = 1
        odd = ans = 0
        for x in nums:
            odd += x & 1
            if odd >= k:
                ans += cnt[odd - k]
            cnt[odd] += 1
        return ans


for nums, k, want in examples:
    assert Solution().numberOfSubarrays(nums, k) == want
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao đúng

**Song ánh:** mảng con `nums[i..j]` (`0 ≤ i ≤ j < n`) ↔ cặp chỉ số prefix `(i, j+1)` với `i < j + 1`. Số lẻ trong đoạn = `p_{j+1} − p_i`.

**Đếm:** khi xử lý phần tử thứ `j` (sau khi cộng, `odd = p_{j+1}`), `cnt[v]` bằng số chỉ số `i ∈ [0, j]` có `p_i = v` (invariant: `cnt[0] = 1` cho `p_0`, và mỗi `p_i` được thêm vào **sau** khi dùng để tra). Cộng `cnt[odd − k]` = số điểm đầu `i` tạo đoạn kết thúc tại `j` có đúng `k` số lẻ. Tổng theo mọi `j` = số mảng con đẹp. Mỗi mảng con được đếm đúng một lần (tại điểm cuối `j` của nó).

**atMost:** tập điểm đầu hợp lệ cho mỗi điểm cuối `r` là một đoạn `[l, r]` (nếu `[l, r]` có ≤ K số lẻ thì mọi đoạn con `[l', r]`, `l' ≥ l`, cũng vậy), và `l` không giảm khi `r` tăng → cửa sổ trượt đúng. Mảng con có đúng `k` số lẻ = (≤ k) trừ (≤ k−1).

**Tổ hợp:** mảng con chứa đúng `k` số lẻ thì tập số lẻ của nó là `k` số lẻ **liên tiếp** `pos[t..t+k−1]`; điểm đầu chạy tự do trong khoảng số chẵn bên trái (không được vượt qua số lẻ trước), điểm cuối tương tự bên phải → tích.

## 11. Time / Space Complexity

Đặt `n = len(nums)`, `m` = số phần tử lẻ.

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n²)` | `O(1)` |
| 2. Prefix + hash map/mảng | `O(n)` | `O(n)` |
| 3. atMost(k) − atMost(k−1) | `O(n)` (2 lượt) | `O(1)` |
| 4. Tổ hợp vị trí | `O(n)` | `O(m)` |
| 5. Một lượt cửa sổ | `O(n)` | `O(1)` |

Đáp án có thể tới `~n²/4 ≈ 6·10⁸` (ví dụ `k` nhỏ, nhiều số chẵn) — Python số nguyên không tràn; ở ngôn ngữ khác cần kiểu 64 bit.

## 12. Bộ test

### 12.1 Ví dụ đề và edge cases

In [16]:
ALL = [SolutionBrute, SolutionPrefixHash, SolutionPrefixArray, SolutionAtMost, SolutionPositions, SolutionOnePass, Solution]


def check(nums, k, want=None):
    outs = [cls().numberOfSubarrays(nums, k) for cls in ALL]
    assert len(set(outs)) == 1, (nums, k, outs)
    if want is not None:
        assert outs[0] == want, (nums, k, outs[0], want)
    return outs[0]


for nums, k, want in examples:
    check(nums, k, want)
check([1], 1, 1)
check([2], 1, 0)
check([1, 1, 1], 3, 1)
check([1, 1, 1], 1, 3)
check([1, 3, 5], 4, 0)                    # k lớn hơn số số lẻ
check([2, 1, 2], 1, 4)                    # [2,1],[1],[1,2],[2,1,2]
check([1, 2, 2, 2, 1], 1, 8)
print("edge cases OK")

edge cases OK


### 12.2 Ngẫu nhiên có seed: đối chiếu với brute force

In [17]:
rng = random.Random(1248)
for _ in range(1000):
    n = rng.randint(1, 25)
    p_odd = rng.choice([0.1, 0.3, 0.5, 0.9])
    nums = [rng.randrange(1, 50, 2) if rng.random() < p_odd else rng.randrange(2, 50, 2) for _ in range(n)]
    k = rng.randint(1, n)
    check(nums, k)
print("1000 test ngẫu nhiên OK")

1000 test ngẫu nhiên OK


### 12.3 Test giới hạn `n = 5·10⁴`

Bỏ brute force (quá chậm với mảng nhiều số chẵn); so các cách `O(n)`.

In [18]:
FAST = ALL[1:]
cases = [
    ([2] * 25000 + [1] + [2] * 24999, 1),                    # đáp án = 25001 * 25000
    ([random.Random(3).randint(1, 10 ** 5) for _ in range(50000)], 100),
    ([1] * 50000, 50000),
]
for nums, k in cases:
    t = time.perf_counter()
    res = [cls().numberOfSubarrays(nums, k) for cls in FAST]
    assert len(set(res)) == 1
    print(f"n={len(nums)}, k={k}: {res[0]} ({(time.perf_counter() - t) * 1000:.0f} ms cho {len(FAST)} cách)")
assert Solution().numberOfSubarrays(cases[0][0], 1) == 25001 * 25000

n=50000, k=1: 625025000 (48 ms cho 6 cách)
n=50000, k=100: 49901 (73 ms cho 6 cách)
n=50000, k=50000: 1 (42 ms cho 6 cách)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `cnt[0] = 1`

In [19]:
def wrong_no_zero(nums, k):
    cnt = Counter()                     # SAI: thiếu prefix rỗng
    odd = ans = 0
    for x in nums:
        odd += x & 1
        ans += cnt[odd - k]
        cnt[odd] += 1
    return ans


print("sai :", wrong_no_zero([1, 1, 2, 1, 1], 3), "| đúng:", Solution().numberOfSubarrays([1, 1, 2, 1, 1], 3))

sai : 1 | đúng: 2


**Vì sao sai:** mảng con bắt đầu từ chỉ số `0` ứng với `p_0 = 0` (prefix rỗng). Thiếu nó thì bỏ sót `[1,1,2,1]`. **Sửa:** `cnt = Counter({0: 1})`.

### Lỗi 2 — Cộng giá trị phần tử thay vì đếm số lẻ

In [20]:
def wrong_sum_values(nums, k):
    cnt = Counter({0: 1})
    s = ans = 0
    for x in nums:
        s += x                          # SAI: tổng giá trị, không phải số lẻ
        ans += cnt[s - k]
        cnt[s] += 1
    return ans


print("sai :", wrong_sum_values([2, 1, 2], 1), "| đúng:", Solution().numberOfSubarrays([2, 1, 2], 1))

sai : 1 | đúng: 4


**Vì sao sai:** đây là lời giải của bài 560 (tổng bằng k), không phải "số lẻ bằng k". `[2,1,2]` có 4 mảng con chứa đúng 1 số lẻ nhưng chỉ 1 mảng con có tổng 1. **Sửa:** cộng `x & 1`.

### Lỗi 3 — Cửa sổ trượt "đúng k" cổ điển, bỏ sót các số chẵn đầu cửa sổ

In [21]:
def wrong_window_exact(nums, k):
    l = odd = ans = 0
    for r, x in enumerate(nums):
        odd += x & 1
        while odd > k:
            odd -= nums[l] & 1
            l += 1
        if odd == k:
            ans += 1                    # SAI: chỉ đếm 1 điểm đầu cho mỗi r
    return ans


nums = [2, 2, 2, 1, 2, 2, 1, 2, 2, 2]
print("sai :", wrong_window_exact(nums, 2), "| đúng:", Solution().numberOfSubarrays(nums, 2))

sai : 4 | đúng: 16


**Vì sao sai:** với mỗi điểm cuối có thể có **nhiều** điểm đầu hợp lệ (các số chẵn ở đầu cửa sổ có thể bỏ hoặc giữ). Co cửa sổ "đúng k" không đơn điệu như "≤ k". **Sửa:** dùng `atMost(k) − atMost(k−1)` hoặc đếm số chẵn dẫn đầu (Cách 5), hoặc prefix + hash map.

### Lỗi 4 — Công thức vị trí quên lính canh ở hai đầu

In [22]:
def wrong_positions(nums, k):
    pos = [i for i, x in enumerate(nums) if x & 1]      # SAI: thiếu -1 và n
    ans = 0
    for t in range(1, len(pos) - k):
        ans += (pos[t] - pos[t - 1]) * (pos[t + k] - pos[t + k - 1])
    return ans


nums = [2, 2, 2, 1, 2, 2, 1, 2, 2, 2]
print("sai :", wrong_positions(nums, 2), "| đúng:", Solution().numberOfSubarrays(nums, 2))

sai : 0 | đúng: 16


**Vì sao sai:** nhóm `k` số lẻ đầu tiên không có "số lẻ trước" và nhóm cuối không có "số lẻ sau" → bị bỏ qua hoàn toàn. **Sửa:** thêm lính canh `pos = [-1] + pos + [n]`.

## 14. Mẹo & nhận dạng pattern

- "Đếm mảng con có **đúng** `k` phần tử thoả tính chất P" → biến thành mảng 0/1 theo P, rồi **prefix sum + hash map** (đếm cặp prefix có hiệu `k`).
- "Đúng k" khó trượt, "nhiều nhất k" dễ trượt → **exactly(k) = atMost(k) − atMost(k−1)**.
- Tính chất "thưa" (ít phần tử đặc biệt) → làm việc trên **danh sách vị trí** + lính canh, đếm bằng tích khoảng cách.
- Khởi tạo `cnt[0] = 1` là "prefix rỗng" — lỗi bị quên nhiều nhất.
- Bài cùng họ: 560 Subarray Sum Equals K, 930 Binary Subarrays With Sum, 992 Subarrays with K Different Integers, 974 Subarray Sums Divisible by K, 2062 Count Vowel Substrings.

## 15. Biến thể

1. **Ít nhất `k` số lẻ** → `total − atMost(k − 1)` với `total = n(n+1)/2`.
2. **Số chẵn bằng số lẻ** → map lẻ → `+1`, chẵn → `−1`, đếm cặp prefix bằng nhau (bài 525 kiểu đếm).
3. **930 Binary Subarrays With Sum** (mảng 0/1, tổng = goal, goal có thể bằng 0) → đúng Cách 2/3; cẩn thận `atMost(−1) = 0`.
4. **992 Subarrays with K Different Integers** → `atMost(k) − atMost(k−1)` với cửa sổ đếm số giá trị khác nhau.
5. **Độ dài mảng con nice ngắn nhất/dài nhất** → dùng danh sách vị trí: ngắn nhất `pos[t+k−1] − pos[t] + 1`, dài nhất `pos[t+k] − pos[t−1] − 1`.

Demo biến thể 1 và 5:

In [23]:
def at_least_k(nums, k):
    def at_most(K):
        if K < 0:
            return 0
        l = odd = res = 0
        for r, x in enumerate(nums):
            odd += x & 1
            while odd > K:
                odd -= nums[l] & 1
                l += 1
            res += r - l + 1
        return res

    n = len(nums)
    return n * (n + 1) // 2 - at_most(k - 1)


def nice_lengths(nums, k):
    pos = [-1] + [i for i, x in enumerate(nums) if x & 1] + [len(nums)]
    shortest = [pos[t + k - 1] - pos[t] + 1 for t in range(1, len(pos) - k)]
    longest = [pos[t + k] - pos[t - 1] - 1 for t in range(1, len(pos) - k)]
    return (min(shortest), max(longest)) if shortest else None


nums = [2, 2, 2, 1, 2, 2, 1, 2, 2, 2]
brute_ge = sum(1 for i in range(len(nums)) for j in range(i, len(nums)) if sum(x & 1 for x in nums[i:j + 1]) >= 1)
print("ít nhất 1 số lẻ:", at_least_k(nums, 1), "| brute:", brute_ge)
print("(ngắn nhất, dài nhất) với k = 2:", nice_lengths(nums, 2))

ít nhất 1 số lẻ: 40 | brute: 40
(ngắn nhất, dài nhất) với k = 2: (4, 10)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Mọi điểm đầu, mở rộng điểm cuối | `O(n²)` | `O(1)` | Kiểm thử |
| 2. Prefix + hash/mảng | Đếm prefix bằng `odd − k` | `O(n)` | `O(n)` | Mặc định, tổng quát (code nộp) |
| 3. atMost hai lần | Cửa sổ trượt ≤ k | `O(n)` | `O(1)` | Khi cần `O(1)` bộ nhớ, pattern quen thuộc |
| 4. Tổ hợp vị trí | Tích khoảng trống hai bên | `O(n)` | `O(m)` | Khi cần thêm thông tin cấu trúc (độ dài...) |
| 5. Một lượt cửa sổ | Đếm số chẵn dẫn đầu | `O(n)` | `O(1)` | Phỏng vấn nâng cao; dễ sai biên |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `nums = [1,2,1,2,1]`, `k = 2`?

**Bài 2 (tính tay):** với `nums = [2,1,2,2,1,2]`, `k = 1`, dùng công thức vị trí tính đáp án.

**Bài 3 (điền TODO):** viết `count_exact_sum01(bits, goal)` cho mảng 0/1 (bài 930), `goal` có thể bằng 0.

**Bài 4 (tự viết lại):** viết `nice_at_most(nums, k)` dùng `atMost` (không nhìn Cách 3).

**Bài 5 (mini challenge):** viết `count_even_eq_odd(nums)` đếm mảng con có số phần tử chẵn bằng số phần tử lẻ.

In [24]:
def count_exact_sum01(bits, goal):
    # TODO: prefix + hash map
    raise NotImplementedError


def nice_at_most(nums, k):
    # TODO
    raise NotImplementedError


def count_even_eq_odd(nums):
    # TODO: lẻ -> +1, chẵn -> -1, đếm cặp prefix bằng nhau
    raise NotImplementedError


# Test gợi ý:
# assert count_exact_sum01([1, 0, 1, 0, 1], 2) == 4
# assert count_exact_sum01([0, 0, 0], 0) == 6
# assert nice_at_most([1, 1, 2, 1, 1], 3) == 2
# assert count_even_eq_odd([1, 2, 3, 4]) == 4

## 18. Gợi ý 3 mức

**Bài 1–2:**
- Mức 1: đánh dấu vị trí số lẻ.
- Mức 2: Bài 1: `pos = [-1, 0, 2, 4, 5]`.
- Mức 3: Bài 2: `pos = [-1, 1, 4, 6]`, `k = 1` → `t = 1`: `2 · 3`; `t = 2`: `3 · 2`.

**Bài 3–4:**
- Mức 1: Bài 3 giống hệt Cách 2 nhưng cộng `b` thay vì `x & 1`.
- Mức 2: `goal = 0` vẫn đúng vì `cnt[odd − 0] = cnt[odd]` đã chứa các prefix trước.
- Mức 3: Bài 4: viết `at_most(K)` trả về 0 khi `K < 0`.

**Bài 5:**
- Mức 1: số chẵn = số lẻ ⇔ tổng (+1/−1) bằng 0.
- Mức 2: đếm cặp prefix có cùng giá trị.
- Mức 3: `ans += cnt[s]; cnt[s] += 1` với `cnt = {0: 1}`.

## 19. Đáp án tham khảo

**Bài 1:** `4` — `[1,2,1]`, `[1,2,1,2]`, `[2,1,2,1]`, `[1,2,1]` (thứ hai). **Bài 2:** `2·3 + 3·2 = 12`.

In [25]:
assert Solution().numberOfSubarrays([1, 2, 1, 2, 1], 2) == 4
assert Solution().numberOfSubarrays([2, 1, 2, 2, 1, 2], 1) == 12
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
def count_exact_sum01(bits, goal):
    cnt = Counter({0: 1})
    s = ans = 0
    for b in bits:
        s += b
        ans += cnt[s - goal]
        cnt[s] += 1
    return ans


def nice_at_most(nums, k):
    def at_most(K):
        if K < 0:
            return 0
        l = odd = res = 0
        for r, x in enumerate(nums):
            odd += x & 1
            while odd > K:
                odd -= nums[l] & 1
                l += 1
            res += r - l + 1
        return res

    return at_most(k) - at_most(k - 1)


def count_even_eq_odd(nums):
    cnt = Counter({0: 1})
    s = ans = 0
    for x in nums:
        s += 1 if x & 1 else -1
        ans += cnt[s]
        cnt[s] += 1
    return ans


assert count_exact_sum01([1, 0, 1, 0, 1], 2) == 4
assert count_exact_sum01([0, 0, 0], 0) == 6
assert nice_at_most([1, 1, 2, 1, 1], 3) == 2
assert count_even_eq_odd([1, 2, 3, 4]) == 4
rng = random.Random(8)
for _ in range(300):
    n = rng.randint(1, 15)
    nums = [rng.randint(1, 9) for _ in range(n)]
    k = rng.randint(1, n)
    assert nice_at_most(nums, k) == SolutionBrute().numberOfSubarrays(nums, k)
    bits = [x & 1 for x in nums]
    g = rng.randint(0, n)
    assert count_exact_sum01(bits, g) == sum(1 for i in range(n) for j in range(i, n) if sum(bits[i:j + 1]) == g)
    assert count_even_eq_odd(nums) == sum(1 for i in range(n) for j in range(i, n)
                                          if 2 * sum(bits[i:j + 1]) == j - i + 1)
print("Bài 3, 4, 5 OK")

Bài 3, 4, 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (prefix + đếm):**

```python
class Solution:
    def numberOfSubarrays(self, nums: List[int], k: int) -> int:
        cnt = [0] * (len(nums) + 1)
        cnt[0] = 1
        odd = ans = 0
        for x in nums:
            odd += x & 1
            if odd >= k:
                ans += cnt[odd - k]
            cnt[odd] += 1
        return ans
```

**Template (exactly = atMost − atMost):** `return at_most(k) - at_most(k - 1)`.

**Checklist trước khi submit:**

- [ ] Đổi phần tử thành `x & 1` (đếm số lẻ, không cộng giá trị).
- [ ] `cnt[0] = 1` cho prefix rỗng.
- [ ] Tra `cnt[odd - k]` **trước** khi tăng `cnt[odd]`.
- [ ] Nếu dùng cửa sổ: không đếm "đúng k" trực tiếp — dùng hiệu hai `atMost`.
- [ ] Công thức vị trí: có lính canh `-1` và `n`.